# Yambda ClassBalance baseline


## 0. Imports and config


In [1]:
import os
import json
import random
import itertools
import gc
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import polars as pl

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm


def set_seed(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


CFG = {
    # Paths
    "data_dir": "../data",
    "interactions_file": "likes.parquet",
    "artist_file": "artist_item_mapping.parquet",
    "album_file": "album_item_mapping.parquet",

    # Debug/data filtering
    # None = all users; for debugging set 50_000 or 100_000
    "max_users": None,
    "min_user_interactions": 3,
    "min_item_interactions": 5,

    # Model
    "embed_dim": 64,
    "artist_emb_dim": 32,
    "album_emb_dim": 32,
    "tower_hidden": [256, 128, 64],
    "dropout": 0.0,

    # Training
    "batch_size": 4096,
    "grad_clip": 1.0,
    "lr": 1e-3,
    "weight_decay": 0.0,

    # Grid search
    "tune_fast": True,
    "tune_epochs": 3,
    "tune_val_fraction": 1.00,
    "skip_tune_if_cached": True,
    "cache_path": "best_hparams_yambda_likes_features.json",

    # Final training
    "final_epochs": 20,

    # Evaluation
    "eval_k": [10, 50],
    "eval_batch_users": 128,
    "eval_item_batch": 8192,
    "head_fraction": 0.20,

    # Reproducibility
    "seed": 0,
    "seeds": [0, 1, 2, 3, 4],
}

set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print(f"Final: {len(CFG['seeds'])} seed(s) × {CFG['final_epochs']} epochs")

device: cuda
Final: 5 seed(s) × 20 epochs


## 1. Load data


In [2]:
def find_file(data_dir: str | Path, name: str) -> Path:
    data_dir = Path(data_dir)
    candidates = [
        data_dir / name,
        data_dir / f"{name}.parquet",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {name}. Tried: {candidates}")


def normalize_interaction_columns(df: pl.DataFrame) -> pl.DataFrame:
    rename = {}
    for c in df.columns:
        lc = c.lower()
        if lc in {"uid", "user_id", "userid", "user"}:
            rename[c] = "uid"
        elif lc in {"item_id", "itemid", "track_id", "trackid", "song_id"}:
            rename[c] = "item_id"
        elif lc in {"timestamp", "ts", "time", "event_timestamp", "datetime"}:
            rename[c] = "timestamp"
    return df.rename(rename)


def normalize_item_side_columns(df: pl.DataFrame) -> pl.DataFrame:
    rename = {}
    for c in df.columns:
        lc = c.lower()
        if lc in {"item_id", "itemid", "track_id", "trackid"}:
            rename[c] = "item_id"
        elif lc in {"artist_id", "artistid"}:
            rename[c] = "artist_id"
        elif lc in {"album_id", "albumid"}:
            rename[c] = "album_id"
    return df.rename(rename)


DATA_DIR = Path(CFG["data_dir"])
INTERACTIONS_PATH = find_file(DATA_DIR, CFG["interactions_file"])
ARTIST_PATH = find_file(DATA_DIR, CFG["artist_file"])
ALBUM_PATH = find_file(DATA_DIR, CFG["album_file"])

print("interactions:", INTERACTIONS_PATH)
print("artists:", ARTIST_PATH)
print("albums:", ALBUM_PATH)

interactions = pl.read_parquet(INTERACTIONS_PATH)
interactions = normalize_interaction_columns(interactions)

print("raw interactions:", interactions.shape)
print("columns:", interactions.columns)
print(interactions.head())

required = {"uid", "item_id"}
missing = required - set(interactions.columns)
assert not missing, f"Missing required columns {missing}. Available: {interactions.columns}"

if "timestamp" not in interactions.columns:
    print("[WARN] timestamp not found: using row index as timestamp")
    interactions = interactions.with_row_index("timestamp")

interactions = interactions.select(["uid", "item_id", "timestamp"])

# Deduplicate repeated likes by the same user for the same item.
interactions = (
    interactions
    .sort("timestamp")
    .unique(subset=["uid", "item_id"], keep="first")
)

print("after dedup:", interactions.shape)

interactions: ../data/likes.parquet
artists: ../data/artist_item_mapping.parquet
albums: ../data/album_item_mapping.parquet
raw interactions: (881456, 4)
columns: ['uid', 'timestamp', 'item_id', 'is_organic']
shape: (5, 4)
┌─────┬───────────┬─────────┬────────────┐
│ uid ┆ timestamp ┆ item_id ┆ is_organic │
│ --- ┆ ---       ┆ ---     ┆ ---        │
│ u32 ┆ u32       ┆ u32     ┆ u8         │
╞═════╪═══════════╪═════════╪════════════╡
│ 100 ┆ 44755     ┆ 732449  ┆ 1          │
│ 100 ┆ 1155860   ┆ 6568592 ┆ 0          │
│ 100 ┆ 1259125   ┆ 5411243 ┆ 1          │
│ 100 ┆ 1260005   ┆ 7371186 ┆ 0          │
│ 100 ┆ 1263935   ┆ 4943655 ┆ 0          │
└─────┴───────────┴─────────┴────────────┘
after dedup: (844690, 3)


## 2. Filtering


In [3]:
# Item-core filter
item_counts = interactions.group_by("item_id").len().rename({"len": "cnt"})
good_items = item_counts.filter(pl.col("cnt") >= CFG["min_item_interactions"]).select("item_id")
interactions = interactions.join(good_items, on="item_id", how="semi")
print("after item-core:", interactions.shape)

# User-core filter
user_counts = interactions.group_by("uid").len().rename({"len": "cnt"})
good_users = user_counts.filter(pl.col("cnt") >= CFG["min_user_interactions"]).select("uid")
interactions = interactions.join(good_users, on="uid", how="semi")
print("after user-core:", interactions.shape)

# Optional debug subset by users. This does not break histories.
if CFG["max_users"] is not None:
    users_sub = (
        interactions
        .select("uid")
        .unique()
        .sample(n=int(CFG["max_users"]), seed=CFG["seed"])
    )
    interactions = interactions.join(users_sub, on="uid", how="semi")
    print(f"after max_users={CFG['max_users']}:", interactions.shape)

    # Repeat filters after subsampling.
    item_counts = interactions.group_by("item_id").len().rename({"len": "cnt"})
    good_items = item_counts.filter(pl.col("cnt") >= CFG["min_item_interactions"]).select("item_id")
    interactions = interactions.join(good_items, on="item_id", how="semi")

    user_counts = interactions.group_by("uid").len().rename({"len": "cnt"})
    good_users = user_counts.filter(pl.col("cnt") >= CFG["min_user_interactions"]).select("uid")
    interactions = interactions.join(good_users, on="uid", how="semi")

print("final filtered:", interactions.shape)
print("users:", interactions["uid"].n_unique())
print("items:", interactions["item_id"].n_unique())

after item-core: (621417, 3)
after user-core: (620105, 3)
final filtered: (620105, 3)
users: 7102
items: 33145


## 3. ID mapping and leave-one-out split


In [4]:
# User/item ids -> contiguous indices.
user_mapping = interactions.select("uid").unique().sort("uid").with_row_index(name="u_idx")
item_mapping = interactions.select("item_id").unique().sort("item_id").with_row_index(name="i_idx")

inter = (
    interactions
    .join(user_mapping, on="uid", how="inner")
    .join(item_mapping, on="item_id", how="inner")
    .select(["u_idx", "i_idx", "timestamp"])
    .sort(["u_idx", "timestamp"])
)

NUM_USERS = user_mapping.height
NUM_ITEMS = item_mapping.height

inter = inter.with_columns([
    pl.arange(0, pl.len()).over("u_idx").alias("pos"),
    pl.len().over("u_idx").alias("hist_len"),
])

# Leave-one-out evaluation with validation:
# most recent item -> test, second most recent -> validation, rest -> train.
train_df = inter.filter(pl.col("pos") < pl.col("hist_len") - 2).select(["u_idx", "i_idx"])
val_df   = inter.filter(pl.col("pos") == pl.col("hist_len") - 2).select(["u_idx", "i_idx"])
test_df  = inter.filter(pl.col("pos") == pl.col("hist_len") - 1).select(["u_idx", "i_idx"])

train_pairs = train_df.to_numpy().astype(np.int64)
val_np = val_df.to_numpy().astype(np.int64)
test_np = test_df.to_numpy().astype(np.int64)

val_u, val_i = val_np[:, 0], val_np[:, 1]
test_u, test_i = test_np[:, 0], test_np[:, 1]

print(f"NUM_USERS={NUM_USERS:,}")
print(f"NUM_ITEMS={NUM_ITEMS:,}")
print(f"train={len(train_pairs):,} val={len(val_u):,} test={len(test_u):,}")
print(f"catalog share @50 = {50 / NUM_ITEMS:.6f}")

assert len(train_pairs) > 0
assert len(val_u) == NUM_USERS
assert len(test_u) == NUM_USERS

NUM_USERS=7,102
NUM_ITEMS=33,145
train=605,901 val=7,102 test=7,102
catalog share @50 = 0.001509


## 4. Build item features: artist_id and album_id


In [5]:
# item_mapping contains original item_id and new i_idx.
# Features are built in i_idx order.
item_features_df = item_mapping.select(["item_id", "i_idx"])


# ---------- artist feature ----------
artists = pl.read_parquet(ARTIST_PATH)
artists = normalize_item_side_columns(artists)

print("artists shape:", artists.shape)
print("artists columns:", artists.columns)

if "item_id" not in artists.columns or "artist_id" not in artists.columns:
    raise ValueError(f"Bad artist mapping columns: {artists.columns}")

# If an item has multiple artists, take the first one for a simple baseline.
artists_one = (
    artists
    .select(["item_id", "artist_id"])
    .drop_nulls(["item_id", "artist_id"])
    .group_by("item_id")
    .agg(pl.col("artist_id").first())
)

item_features_df = item_features_df.join(artists_one, on="item_id", how="left")


# ---------- album feature ----------
albums = pl.read_parquet(ALBUM_PATH)
albums = normalize_item_side_columns(albums)

print("albums shape:", albums.shape)
print("albums columns:", albums.columns)

if "item_id" not in albums.columns or "album_id" not in albums.columns:
    raise ValueError(f"Bad album mapping columns: {albums.columns}")

# If an item has multiple albums, take the first one for a simple baseline.
albums_one = (
    albums
    .select(["item_id", "album_id"])
    .drop_nulls(["item_id", "album_id"])
    .group_by("item_id")
    .agg(pl.col("album_id").first())
)

item_features_df = item_features_df.join(albums_one, on="item_id", how="left")


# ---------- categorical encoding ----------
item_features_df = item_features_df.sort("i_idx")

# unknown = 0, real categories = 1..N
unique_artists = (
    item_features_df
    .select("artist_id")
    .drop_nulls()
    .unique()
    .sort("artist_id")
    .with_row_index(name="artist_idx", offset=1)
)

item_features_df = (
    item_features_df
    .join(unique_artists, on="artist_id", how="left")
    .with_columns(pl.col("artist_idx").fill_null(0).cast(pl.Int64))
)

unique_albums = (
    item_features_df
    .select("album_id")
    .drop_nulls()
    .unique()
    .sort("album_id")
    .with_row_index(name="album_idx", offset=1)
)

item_features_df = (
    item_features_df
    .join(unique_albums, on="album_id", how="left")
    .with_columns(pl.col("album_idx").fill_null(0).cast(pl.Int64))
)

ITEM_ARTIST = item_features_df["artist_idx"].to_numpy().astype(np.int64)
ITEM_ALBUM = item_features_df["album_idx"].to_numpy().astype(np.int64)

NUM_ARTISTS = int(ITEM_ARTIST.max()) + 1
NUM_ALBUMS = int(ITEM_ALBUM.max()) + 1

print("NUM_ITEMS:", NUM_ITEMS)
print("NUM_ARTISTS:", NUM_ARTISTS)
print("NUM_ALBUMS:", NUM_ALBUMS)
print("artist unknown share:", float((ITEM_ARTIST == 0).mean()))
print("album unknown share:", float((ITEM_ALBUM == 0).mean()))

item_artist_t = torch.from_numpy(ITEM_ARTIST).long().to(device)
item_album_t = torch.from_numpy(ITEM_ALBUM).long().to(device)

artists shape: (9271906, 2)
artists columns: ['artist_id', 'item_id']
albums shape: (9651644, 2)
albums columns: ['album_id', 'item_id']
NUM_ITEMS: 33145
NUM_ARTISTS: 9321
NUM_ALBUMS: 23839
artist unknown share: 0.0
album unknown share: 3.017046311660884e-05


## 5. Known items and head/tail split


In [6]:
train_user_items = [set() for _ in range(NUM_USERS)]

for u, i in train_pairs:
    train_user_items[int(u)].add(int(i))

known_val = [set(s) for s in train_user_items]
known_test = [set(s) for s in train_user_items]

# For test, validation item is already known and should be masked.
for u, i in zip(val_u, val_i):
    known_test[int(u)].add(int(i))

# Head/tail is computed only from train frequencies.
item_freq = np.bincount(train_pairs[:, 1], minlength=NUM_ITEMS)
order = np.argsort(-item_freq)

n_head = max(1, int(NUM_ITEMS * CFG["head_fraction"]))
head_mask = np.zeros(NUM_ITEMS, dtype=bool)
head_mask[order[:n_head]] = True

print(f"head={head_mask.sum():,} tail={(~head_mask).sum():,}")
print(f"nonzero train items={np.sum(item_freq > 0):,}")
print(f"zero train items={np.sum(item_freq == 0):,}")
print(f"val true head share={head_mask[val_i].mean():.4f}")
print(f"test true head share={head_mask[test_i].mean():.4f}")

head=6,629 tail=26,516
nonzero train items=33,145
zero train items=0
val true head share=0.5628
test true head share=0.5338


## 6. Dataset and model


In [7]:
class PairDataset(Dataset):
    def __init__(self, pairs: np.ndarray):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        u, i = self.pairs[idx]
        return int(u), int(i)


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden: list[int], dropout: float = 0.0):
        super().__init__()

        layers = []
        d = in_dim

        for h in hidden:
            layers.append(nn.Linear(d, h))
            layers.append(nn.ReLU())

            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            d = h

        self.net = nn.Sequential(*layers)
        self.out_dim = d

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TwoTower(nn.Module):
    """
    Two-Tower baseline:
      user tower: user_id -> MLP
      item tower: item_id + artist_id + album_id -> MLP
    """
    def __init__(
        self,
        num_users: int,
        num_items: int,
        num_artists: int,
        num_albums: int,
        embed_dim: int = 64,
        artist_emb_dim: int = 32,
        album_emb_dim: int = 32,
        hidden: list[int] = [256, 128, 64],
        dropout: float = 0.0,
    ):
        super().__init__()

        self.user_emb = nn.Embedding(num_users, embed_dim)

        self.item_emb = nn.Embedding(num_items, embed_dim)
        self.artist_emb = nn.Embedding(num_artists, artist_emb_dim)
        self.album_emb = nn.Embedding(num_albums, album_emb_dim)

        user_in_dim = embed_dim
        item_in_dim = embed_dim + artist_emb_dim + album_emb_dim

        self.user_mlp = MLP(user_in_dim, hidden, dropout)
        self.item_mlp = MLP(item_in_dim, hidden, dropout)

        self.user_ln = nn.LayerNorm(self.user_mlp.out_dim)
        self.item_ln = nn.LayerNorm(self.item_mlp.out_dim)

        self.init_weights()

    def init_weights(self):
        for emb in [self.user_emb, self.item_emb, self.artist_emb, self.album_emb]:
            nn.init.normal_(emb.weight, std=0.01)

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def user_vec(self, u: torch.Tensor) -> torch.Tensor:
        x = self.user_emb(u)
        x = self.user_mlp(x)
        x = self.user_ln(x)
        return x

    def item_vec(self, i: torch.Tensor) -> torch.Tensor:
        item_id_vec = self.item_emb(i)
        artist_vec = self.artist_emb(item_artist_t[i])
        album_vec = self.album_emb(item_album_t[i])

        x = torch.cat([item_id_vec, artist_vec, album_vec], dim=-1)
        x = self.item_mlp(x)
        x = self.item_ln(x)
        return x

    def forward(self, u: torch.Tensor, i: torch.Tensor) -> torch.Tensor:
        uv = F.normalize(self.user_vec(u), dim=-1, eps=1e-6)
        iv = F.normalize(self.item_vec(i), dim=-1, eps=1e-6)
        return (uv * iv).sum(dim=-1)


def inbatch_softmax_loss(user_vecs: torch.Tensor, item_vecs: torch.Tensor):
    u = F.normalize(user_vecs, dim=-1, eps=1e-6)
    v = F.normalize(item_vecs, dim=-1, eps=1e-6)

    logits = u @ v.T
    labels = torch.arange(logits.size(0), device=logits.device)

    return F.cross_entropy(logits, labels)


train_loader = DataLoader(
    PairDataset(train_pairs),
    batch_size=CFG["batch_size"],
    shuffle=True,
    drop_last=True,
    pin_memory=torch.cuda.is_available(),
)

model = TwoTower(
    NUM_USERS,
    NUM_ITEMS,
    NUM_ARTISTS,
    NUM_ALBUMS,
    embed_dim=CFG["embed_dim"],
    artist_emb_dim=CFG["artist_emb_dim"],
    album_emb_dim=CFG["album_emb_dim"],
    hidden=CFG["tower_hidden"],
    dropout=CFG["dropout"],
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
)

print(model)

TwoTower(
  (user_emb): Embedding(7102, 64)
  (item_emb): Embedding(33145, 64)
  (artist_emb): Embedding(9321, 32)
  (album_emb): Embedding(23839, 32)
  (user_mlp): MLP(
    (net): Sequential(
      (0): Linear(in_features=64, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=128, bias=True)
      (3): ReLU()
      (4): Linear(in_features=128, out_features=64, bias=True)
      (5): ReLU()
    )
  )
  (item_mlp): MLP(
    (net): Sequential(
      (0): Linear(in_features=128, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=128, bias=True)
      (3): ReLU()
      (4): Linear(in_features=128, out_features=64, bias=True)
      (5): ReLU()
    )
  )
  (user_ln): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (item_ln): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
)


## 7. Evaluation


In [8]:
@torch.no_grad()
def evaluate_full_corpus(
    model: nn.Module,
    users: np.ndarray,
    true_items: np.ndarray,
    known_user_items: List[set],
    head_mask: np.ndarray,
    ks: List[int],
    item_freq: np.ndarray,
    user_batch_size: int = 128,
    item_batch_size: int = 8192,
):
    model.eval()

    ranks_all = []
    ranks_head = []
    ranks_tail = []

    max_k = max(ks)

    # Для coverage / popularity / personalization
    recommended_by_k = {k: [] for k in ks}

    # ============================================================
    # 1. Считаем вектора всех items один раз
    # ============================================================

    item_vectors = []

    for s in tqdm(
        range(0, NUM_ITEMS, item_batch_size),
        desc="item vectors",
        leave=False,
    ):
        e = min(s + item_batch_size, NUM_ITEMS)
        idx = torch.arange(s, e, device=device)

        v = model.item_vec(idx)
        v = F.normalize(v, dim=-1, eps=1e-6)

        if not torch.isfinite(v).all():
            raise RuntimeError(f"Non-finite item vectors on item batch {s}:{e}")

        item_vectors.append(v.cpu())

    item_vectors = torch.cat(item_vectors, dim=0).to(device)

    if not torch.isfinite(item_vectors).all():
        raise RuntimeError("Non-finite item vectors after concat")

    # ============================================================
    # 2. Идём по пользователям батчами
    # ============================================================

    for start in tqdm(
        range(0, len(users), user_batch_size),
        desc="eval users",
        leave=False,
    ):
        end = min(start + user_batch_size, len(users))

        bu = users[start:end]
        bi = true_items[start:end]

        ut = torch.tensor(bu, device=device, dtype=torch.long)

        uvec = model.user_vec(ut)
        uvec = F.normalize(uvec, dim=-1, eps=1e-6)

        if not torch.isfinite(uvec).all():
            raise RuntimeError(f"Non-finite user vectors on user batch {start}:{end}")

        scores = (uvec @ item_vectors.T).cpu().numpy()

        if not np.isfinite(scores).all():
            bad = np.sum(~np.isfinite(scores))
            raise RuntimeError(
                f"Non-finite scores in user batch {start}:{end}: {bad} bad values"
            )

        # ========================================================
        # 3. Для каждого пользователя считаем rank и top-K
        # ========================================================

        for row, (u, true_i) in enumerate(zip(bu, bi)):
            u = int(u)
            true_i = int(true_i)

            srow = scores[row].copy()

            if true_i < 0 or true_i >= NUM_ITEMS:
                raise RuntimeError(
                    f"true_i out of bounds: user={u}, true_i={true_i}, NUM_ITEMS={NUM_ITEMS}"
                )

            if not np.isfinite(srow[true_i]):
                raise RuntimeError(
                    f"Non-finite true item score: user={u}, item={true_i}"
                )

            # Маскируем уже известные items пользователя
            for it in known_user_items[u]:
                if it != true_i:
                    srow[it] = -1e9

            if not np.isfinite(srow[true_i]):
                raise RuntimeError(
                    f"True item was masked or became non-finite: user={u}, item={true_i}"
                )

            # ----------------------------------------------------
            # Rank true item
            # pessimistic tie handling:
            # если у многих items одинаковый score, true item НЕ считается первым
            # ----------------------------------------------------

            true_score = srow[true_i]
            eps = 1e-12

            num_greater = int((srow > true_score + eps).sum())
            num_tied = int((np.abs(srow - true_score) <= eps).sum())

            rank = num_greater + num_tied - 1

            ranks_all.append(rank)

            if head_mask[true_i]:
                ranks_head.append(rank)
            else:
                ranks_tail.append(rank)

            # ----------------------------------------------------
            # Top-K recommendations для coverage/popularity
            # ----------------------------------------------------

            if max_k < len(srow):
                top_unsorted = np.argpartition(-srow, max_k - 1)[:max_k]
                top_idx = top_unsorted[np.argsort(-srow[top_unsorted])]
            else:
                top_idx = np.argsort(-srow)

            for k in ks:
                recommended_by_k[k].append(top_idx[:k].astype(np.int64))

    # ============================================================
    # 4. Accuracy metrics: HR / NDCG
    # ============================================================

    def agg_accuracy(ranks: List[int]) -> Dict[str, float]:
        a = np.asarray(ranks, dtype=np.int64)
        out = {}

        for k in ks:
            if len(a) == 0:
                out[f"HR@{k}"] = np.nan
                out[f"NDCG@{k}"] = np.nan
            else:
                hits = a < k

                out[f"HR@{k}"] = 100.0 * hits.mean()

                out[f"NDCG@{k}"] = 100.0 * np.where(
                    hits,
                    1.0 / np.log2(a + 2),
                    0.0,
                ).mean()

        return out

    # ============================================================
    # 5. Long-tail / coverage metrics
    # ============================================================

    tail_mask = ~head_mask
    num_tail_items = int(tail_mask.sum())

    # item_freq нужен именно здесь
    popularity = np.log1p(item_freq.astype(np.float64))

    long_tail_metrics = {}

    for k in ks:
        recs = np.vstack(recommended_by_k[k])  # shape: (num_eval_users, k)

        unique_recs = np.unique(recs)

        # CatalogCoverage@K
        catalog_coverage = len(unique_recs) / NUM_ITEMS

        # TailCoverage@K
        if num_tail_items > 0:
            tail_coverage = np.sum(tail_mask[unique_recs]) / num_tail_items
        else:
            tail_coverage = np.nan

        # AvgPopularity@K
        avg_popularity = popularity[recs].mean()

        # TailRatio@K
        tail_ratio = tail_mask[recs].mean()

        # Personalization@K
        # 1 - average pairwise overlap / K
        # считаем эффективно через exposure counts
        n_users_eval = recs.shape[0]

        if n_users_eval <= 1:
            personalization = np.nan
        else:
            exposure_counts = np.bincount(
                recs.reshape(-1),
                minlength=NUM_ITEMS,
            )

            pairwise_intersections = np.sum(
                exposure_counts * (exposure_counts - 1) / 2
            )

            num_user_pairs = n_users_eval * (n_users_eval - 1) / 2
            avg_overlap = pairwise_intersections / num_user_pairs

            personalization = 1.0 - avg_overlap / k

        long_tail_metrics[f"CatalogCoverage@{k}"] = 100.0 * catalog_coverage
        long_tail_metrics[f"TailCoverage@{k}"] = 100.0 * tail_coverage
        long_tail_metrics[f"AvgPopularity@{k}"] = float(avg_popularity)
        long_tail_metrics[f"TailRatio@{k}"] = 100.0 * tail_ratio
        long_tail_metrics[f"Personalization@{k}"] = 100.0 * personalization

    return {
        "overall": agg_accuracy(ranks_all),
        "head": agg_accuracy(ranks_head),
        "tail": agg_accuracy(ranks_tail),
        "long_tail": long_tail_metrics,
        "counts": {
            "overall": len(ranks_all),
            "head": len(ranks_head),
            "tail": len(ranks_tail),
        },
    }


def print_metrics(metrics: Dict):
    print("counts:", metrics.get("counts", {}))

    for split in ["overall", "head", "tail"]:
        print(f"[{split}]", metrics[split])

    if "long_tail" in metrics:
        print("[long_tail]", metrics["long_tail"])


In [9]:
def make_head_mask(item_freq: np.ndarray, head_fraction: float) -> np.ndarray:
    num_items = len(item_freq)
    n_head = max(1, int(num_items * head_fraction))

    order = np.argsort(-item_freq)

    current_head_mask = np.zeros(num_items, dtype=bool)
    current_head_mask[order[:n_head]] = True

    return current_head_mask


@torch.no_grad()
def evaluate_head_tail_sweep(
    model: nn.Module,
    method_name: str,
    seed: int,
    head_fractions: List[float],
    test_u: np.ndarray,
    test_i: np.ndarray,
    known_test: List[set],
    item_freq: np.ndarray,
    ks: List[int],
):
    rows = []

    model.eval()

    for hf in head_fractions:
        print("\n" + "=" * 80)
        print(f"{method_name} | seed={seed} | head_fraction={hf} ({100 * hf:.3f}%)")
        print("=" * 80)

        current_head_mask = make_head_mask(
            item_freq=item_freq,
            head_fraction=hf,
        )

        num_head_items = int(current_head_mask.sum())
        num_tail_items = int((~current_head_mask).sum())

        test_head_share = float(current_head_mask[test_i].mean())
        test_tail_share = float((~current_head_mask[test_i]).mean())

        print(f"num_head_items: {num_head_items:,}")
        print(f"num_tail_items: {num_tail_items:,}")
        print(f"test_head_share: {test_head_share:.4f}")
        print(f"test_tail_share: {test_tail_share:.4f}")

        metrics = evaluate_full_corpus(
            model=model,
            users=test_u,
            true_items=test_i,
            known_user_items=known_test,
            head_mask=current_head_mask,
            ks=ks,
            item_freq=item_freq,
            user_batch_size=CFG["eval_batch_users"],
            item_batch_size=CFG["eval_item_batch"],
        )

        print_metrics(metrics)

        row = {
            "method": method_name,
            "seed": seed,
            "head_fraction": hf,
            "head_percent": 100.0 * hf,
            "num_head_items": num_head_items,
            "num_tail_items": num_tail_items,
            "test_head_share": test_head_share,
            "test_tail_share": test_tail_share,
        }

        for split in ("overall", "head", "tail"):
            for key, value in metrics[split].items():
                row[f"{split}_{key}"] = value

        if "long_tail" in metrics:
            for key, value in metrics["long_tail"].items():
                row[key] = value

        if "counts" in metrics:
            for key, value in metrics["counts"].items():
                row[f"count_{key}"] = value

        rows.append(row)

    return rows

In [10]:
# Override method-specific config
CFG["cache_path"] = "best_hparams_yambda_classbalance.json"
CFG["cb_beta"] = 0.9999

## 8. ClassBalance loss

In [11]:
def build_classbalance_weights(item_freq: np.ndarray, beta: float = 0.9999) -> torch.Tensor:
    freq = item_freq.astype(np.float64)
    weights = np.zeros_like(freq, dtype=np.float64)
    seen = freq > 0

    weights[seen] = (1.0 - beta) / (1.0 - np.power(beta, freq[seen]))

    # Normalize so that average weight over train positives is close to 1.
    denom = np.sum(freq[seen] * weights[seen]) / np.sum(freq[seen])
    weights[seen] = weights[seen] / max(denom, 1e-12)

    return torch.from_numpy(weights.astype(np.float32))


cb_item_weights = build_classbalance_weights(item_freq, beta=CFG["cb_beta"]).to(device)

print("ClassBalance weights")
print("  beta:", CFG["cb_beta"])
print("  min seen:", float(cb_item_weights[cb_item_weights > 0].min().item()))
print("  max seen:", float(cb_item_weights.max().item()))
print("  mean over train positives:", float(cb_item_weights[torch.from_numpy(train_pairs[:, 1]).long().to(device)].mean().item()))


def classbalance_inbatch_softmax_loss(
    user_vecs: torch.Tensor,
    item_vecs: torch.Tensor,
    pos_items: torch.Tensor,
    item_weights: torch.Tensor,
):
    u = F.normalize(user_vecs, dim=-1, eps=1e-6)
    v = F.normalize(item_vecs, dim=-1, eps=1e-6)

    logits = u @ v.T
    labels = torch.arange(logits.size(0), device=logits.device)

    per_example_loss = F.cross_entropy(logits, labels, reduction="none")
    w = item_weights[pos_items].detach()

    return (per_example_loss * w).sum() / w.sum().clamp_min(1e-12)


ClassBalance weights
  beta: 0.9999
  min seen: 0.025406619533896446
  max seen: 18.264514923095703
  mean over train positives: 1.0


## 9. Grid search

In [12]:
LR_GRID = [0.01, 0.001, 0.0001]
DROPOUT_GRID = [0.1, 0.3, 0.7]
WD_GRID = [0.0, 0.1, 0.001]

combos = list(itertools.product(LR_GRID, DROPOUT_GRID, WD_GRID))
k_main = CFG["eval_k"][-1]

cache_path = Path(CFG["cache_path"])
leaderboard_csv_path = cache_path.with_suffix(".leaderboard.csv")

if CFG["skip_tune_if_cached"] and cache_path.exists():
    best_hp = json.loads(cache_path.read_text())
    print(f"Loaded cached hparams: {best_hp}")
    leaderboard_df = pd.read_csv(leaderboard_csv_path) if leaderboard_csv_path.exists() else None
else:
    frac = float(CFG.get("tune_val_fraction", 1.0))
    if frac < 1.0 and CFG["tune_fast"]:
        n_tune = max(1, int(len(val_u) * frac))
        _idx = np.random.default_rng(42).choice(len(val_u), n_tune, replace=False)
        val_u_t, val_i_t = val_u[_idx], val_i[_idx]
    else:
        val_u_t, val_i_t = val_u, val_i

    tune_ep = CFG["tune_epochs"] if CFG["tune_fast"] else CFG["final_epochs"]

    print(f"Tuning ClassBalance {len(combos)} trials × {tune_ep} ep (val subset: {len(val_u_t):,}/{len(val_u):,})")

    best_hr = -1.0
    best_hp = None
    leaderboard = []

    for ti, (lr, dr, wd) in enumerate(combos, 1):
        set_seed(CFG["seed"])

        m = TwoTower(
            NUM_USERS, NUM_ITEMS, NUM_ARTISTS, NUM_ALBUMS,
            embed_dim=CFG["embed_dim"],
            artist_emb_dim=CFG["artist_emb_dim"],
            album_emb_dim=CFG["album_emb_dim"],
            hidden=CFG["tower_hidden"],
            dropout=dr,
        ).to(device)

        opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=wd)

        status = "ok"
        error_message = ""
        avg_loss = np.nan
        met = None
        hr = -1.0

        try:
            for ep in range(1, tune_ep + 1):
                m.train()
                total_loss, total_n = 0.0, 0
                pbar = tqdm(train_loader, desc=f"CB trial{ti} {ep}/{tune_ep}", leave=False)

                for users_batch, items_batch in pbar:
                    users_batch = users_batch.to(device, non_blocking=True)
                    items_batch = items_batch.to(device, non_blocking=True)

                    user_vecs = m.user_vec(users_batch)
                    item_vecs = m.item_vec(items_batch)

                    loss = classbalance_inbatch_softmax_loss(
                        user_vecs=user_vecs,
                        item_vecs=item_vecs,
                        pos_items=items_batch,
                        item_weights=cb_item_weights,
                    )

                    if not torch.isfinite(loss):
                        raise RuntimeError(f"Non-finite loss: {loss.item()}")

                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(m.parameters(), CFG["grad_clip"])
                    opt.step()

                    bs = users_batch.size(0)
                    total_loss += loss.item() * bs
                    total_n += bs
                    pbar.set_postfix(loss=f"{loss.item():.4f}")

                avg_loss = total_loss / max(total_n, 1)
                print(f"  CB trial{ti} ep{ep} loss={avg_loss:.4f}")

            met = evaluate_full_corpus(
                model=m,
                users=val_u_t,
                true_items=val_i_t,
                known_user_items=known_val,
                head_mask=head_mask,
                ks=CFG["eval_k"],
                item_freq=item_freq,
                user_batch_size=CFG["eval_batch_users"],
                item_batch_size=CFG["eval_item_batch"],
            )
            hr = met["overall"][f"HR@{k_main}"]
            if not np.isfinite(hr):
                raise RuntimeError(f"Non-finite HR: {hr}")
        except RuntimeError as e:
            status = "failed"
            error_message = str(e)
            print(f"  CB trial {ti} FAILED: {e}")

        row = {
            "trial": ti,
            "status": status,
            "error": error_message,
            "method": "ClassBalance",
            "lr": lr,
            "dropout": dr,
            "weight_decay": wd,
            "cb_beta": CFG["cb_beta"],
            "tune_epochs": tune_ep,
            "val_subset_size": len(val_u_t),
            "val_full_size": len(val_u),
            "final_train_loss": avg_loss,
            f"val_HR@{k_main}": hr,
        }

        if met is not None:
            for split in ("overall", "head", "tail"):
                for key, value in met[split].items():
                    row[f"val_{split}_{key}"] = value
            if "long_tail" in met:
                for key, value in met["long_tail"].items():
                    row[f"val_{key}"] = value
            if "counts" in met:
                for key, value in met["counts"].items():
                    row[f"val_count_{key}"] = value

        leaderboard.append(row)
        print(f"  CB trial {ti:3d}/{len(combos)} lr={lr:.0e} dr={dr} wd={wd:.0e} -> val HR@{k_main}={hr:.2f}%")

        if status == "ok" and hr > best_hr:
            best_hr = hr
            best_hp = {"lr": lr, "dropout": dr, "weight_decay": wd, "cb_beta": CFG["cb_beta"], f"val_HR@{k_main}": hr}

        del m, opt
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    leaderboard_df = pd.DataFrame(leaderboard).sort_values(f"val_HR@{k_main}", ascending=False, na_position="last")
    leaderboard_df.to_csv(leaderboard_csv_path, index=False)

    if best_hp is None:
        raise RuntimeError("No successful grid-search trial. Check leaderboard for errors.")

    cache_path.write_text(json.dumps(best_hp, indent=2))
    print(f"\nBest ClassBalance val HR@{k_main}={best_hr:.2f}% -> {best_hp}")
    print(f"Saved best hparams: {cache_path}")
    print(f"Saved leaderboard CSV: {leaderboard_csv_path}")

leaderboard_df.head(10) if leaderboard_df is not None else None


Tuning ClassBalance 80 trials × 3 ep (val subset: 7,102/7,102)


  CB trial1 ep1 loss=7.9529


  CB trial1 ep2 loss=7.9090


  CB trial1 ep3 loss=7.9068


  CB trial   1/80 lr=1e-01 dr=0.1 wd=0e+00 -> val HR@50=0.07%


...

  CB trial80 ep1 loss=8.3274


  CB trial80 ep2 loss=8.3269


  CB trial80 ep3 loss=8.3260


  CB trial  80/80 lr=1e-04 dr=0.9 wd=1e-03 -> val HR@50=0.10%

Best ClassBalance val HR@50=0.35% -> {'lr': 0.001, 'dropout': 0.9, 'weight_decay': 0.001, 'cb_beta': 0.9999, 'val_HR@50': 0.35201351731906505}
Saved best hparams: best_hparams_yambda_classbalance.json
Saved leaderboard CSV: best_hparams_yambda_classbalance.leaderboard.csv


,trial,status,error,method,lr,dropout,weight_decay,cb_beta,tune_epochs,val_subset_size,...,val_TailRatio@10,val_Personalization@10,val_CatalogCoverage@50,val_TailCoverage@50,val_AvgPopularity@50,val_TailRatio@50,val_Personalization@50,val_count_overall,val_count_head,val_count_tail
59,60,ok,,ClassBalance,0.0010,0.9,0.001,0.9999,3,7102,...,30.677274,2.509342,0.214210,0.154624,3.300152,50.675303,1.484856,7102,3997,3105
40,41,ok,,ClassBalance,0.0010,0.1,0.000,0.9999,3,7102,...,100.000000,94.483854,16.240760,20.300950,1.974728,100.000000,93.027217,7102,3997,3105
62,63,ok,,ClassBalance,0.0001,0.1,0.010,0.9999,3,7102,...,100.000000,0.134825,0.187057,0.233821,1.749327,100.000000,1.535848,7102,3997,3105
20,21,ok,,ClassBalance,0.0100,0.1,0.000,0.9999,3,7102,...,99.985919,96.779863,25.859104,32.320109,1.973930,99.975218,96.147464,7102,3997,3105
48,49,ok,,ClassBalance,0.0010,0.5,0.000,0.9999,3,7102,...,91.854407,26.274536,4.564791,5.457083,2.216225,94.728809,28.468681,7102,3997,3105
78,79,ok,,ClassBalance,0.0001,0.9,0.010,0.9999,3,7102,...,90.088707,0.464009,0.199125,0.192337,2.663537,80.106449,0.544779,7102,3997,3105
32,33,ok,,ClassBalance,0.0100,0.7,0.000,0.9999,3,7102,...,100.000000,0.359783,0.165938,0.199879,2.504666,96.016333,0.468422,7102,3997,3105
28,29,ok,,ClassBalance,0.0100,0.5,0.000,0.9999,3,7102,...,100.000000,3.196007,0.165938,0.207422,2.023971,100.000000,1.869088,7102,3997,3105
56,57,ok,,ClassBalance,0.0010,0.9,0.000,0.9999,3,7102,...,79.923965,0.345852,0.159903,0.165938,2.437570,84.118840,0.424939,7102,3997,3105
77,78,ok,,ClassBalance,0.0001,0.9,0.100,0.9999,3,7102,...,60.283019,0.683068,0.174989,0.177251,2.471482,82.096593,0.398292,7102,3997,3105


## 10. Final training over seeds + head/tail sweep

In [13]:

HEAD_FRACTIONS = [0.001, 0.005, 0.01, 0.05, 0.20]

all_rows = []
all_test = []
all_sweep_rows = []

print("Using best_hp:", best_hp)

for seed in CFG["seeds"]:
    print("\n" + "=" * 80)
    print(f"ClassBalance seed {seed}")
    print("=" * 80)

    set_seed(seed)

    model = TwoTower(
        NUM_USERS, NUM_ITEMS, NUM_ARTISTS, NUM_ALBUMS,
        embed_dim=CFG["embed_dim"],
        artist_emb_dim=CFG["artist_emb_dim"],
        album_emb_dim=CFG["album_emb_dim"],
        hidden=CFG["tower_hidden"],
        dropout=best_hp["dropout"],
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=best_hp["lr"], weight_decay=best_hp["weight_decay"])

    best_val_hr50 = -1.0
    best_state = None

    for epoch in range(1, CFG["final_epochs"] + 1):
        model.train()
        total_loss, total_n = 0.0, 0
        pbar = tqdm(train_loader, desc=f"CB seed {seed} train {epoch}/{CFG['final_epochs']}", leave=False)

        for users_batch, items_batch in pbar:
            users_batch = users_batch.to(device, non_blocking=True)
            items_batch = items_batch.to(device, non_blocking=True)

            user_vecs = model.user_vec(users_batch)
            item_vecs = model.item_vec(items_batch)

            loss = classbalance_inbatch_softmax_loss(user_vecs, item_vecs, items_batch, cb_item_weights)

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss at seed={seed}, epoch={epoch}: {loss.item()}")

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            optimizer.step()

            bs = users_batch.size(0)
            total_loss += loss.item() * bs
            total_n += bs
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_loss = total_loss / max(total_n, 1)
        print(f"seed {seed} epoch {epoch}: train loss = {avg_loss:.4f}")

        val_metrics = evaluate_full_corpus(
            model=model,
            users=val_u,
            true_items=val_i,
            known_user_items=known_val,
            head_mask=head_mask,
            ks=CFG["eval_k"],
            item_freq=item_freq,
            user_batch_size=CFG["eval_batch_users"],
            item_batch_size=CFG["eval_item_batch"],
        )

        hr50 = val_metrics["overall"].get("HR@50", -1.0)
        if hr50 > best_val_hr50:
            best_val_hr50 = hr50
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f"new best val HR@50 = {best_val_hr50:.4f}")

    assert best_state is not None
    model.load_state_dict(best_state)
    model.to(device)
    model.eval()

    test_metrics = evaluate_full_corpus(
        model=model,
        users=test_u,
        true_items=test_i,
        known_user_items=known_test,
        head_mask=head_mask,
        ks=CFG["eval_k"],
        item_freq=item_freq,
        user_batch_size=CFG["eval_batch_users"],
        item_batch_size=CFG["eval_item_batch"],
    )

    print("TEST METRICS")
    print_metrics(test_metrics)
    all_test.append(test_metrics)

    row = {
        "method": "ClassBalance",
        "seed": seed,
        "lr": best_hp["lr"],
        "dropout": best_hp["dropout"],
        "weight_decay": best_hp["weight_decay"],
        "cb_beta": best_hp.get("cb_beta", CFG["cb_beta"]),
        "best_val_HR@50": best_val_hr50,
    }
    for split in ("overall", "head", "tail"):
        for key, value in test_metrics[split].items():
            row[f"test_{split}_{key}"] = value
    if "long_tail" in test_metrics:
        for key, value in test_metrics["long_tail"].items():
            row[f"test_{key}"] = value
    if "counts" in test_metrics:
        for key, value in test_metrics["counts"].items():
            row[f"test_count_{key}"] = value
    all_rows.append(row)

    sweep_rows_seed = evaluate_head_tail_sweep(
        model=model,
        method_name="ClassBalance",
        seed=seed,
        head_fractions=HEAD_FRACTIONS,
        test_u=test_u,
        test_i=test_i,
        known_test=known_test,
        item_freq=item_freq,
        ks=CFG["eval_k"],
    )
    all_sweep_rows.extend(sweep_rows_seed)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

metrics_df = pd.DataFrame(all_rows)
sweep_df = pd.DataFrame(all_sweep_rows)
metrics_df


Using best_hp: {'lr': 0.001, 'dropout': 0.9, 'weight_decay': 0.001, 'cb_beta': 0.9999, 'val_HR@50': 0.35201351731906505}

ClassBalance seed 0


seed 0 epoch 1: train loss = 8.3249


new best val HR@50 = 0.0986


seed 0 epoch 2: train loss = 8.3194


new best val HR@50 = 0.1126


...

counts: {'overall': 7102, 'head': 454, 'tail': 6648}
[overall] {'HR@10': 0.042241622078287806, 'NDCG@10': 0.01999429146563426, 'HR@50': 0.23936919177696422, 'NDCG@50': 0.06550035783845347}
[head] {'HR@10': 0.0, 'NDCG@10': 0.0, 'HR@50': 1.762114537444934, 'NDCG@50': 0.40365718816729684}
[tail] {'HR@10': 0.04512635379061372, 'NDCG@10': 0.021359725930946833, 'HR@50': 0.13537906137184114, 'NDCG@50': 0.04240721689842716}
[long_tail] {'CatalogCoverage@10': 0.04525569467491326, 'TailCoverage@10': 0.04244996967859309, 'AvgPopularity@10': 3.109376966817976, 'TailRatio@10': 99.55927907631653, 'Personalization@10': 14.56535030564945, 'CatalogCoverage@50': 0.25946598280283606, 'TailCoverage@50': 0.25166767738023044, 'AvgPopularity@50': 2.857488557061844, 'TailRatio@50': 94.32807659814137, 'Personalization@50': 7.885432662436509}

ClassBalance | seed=4 | head_fraction=0.01 (1.000%)
num_head_items: 331
num_tail_items: 32,814
test_head_share: 0.1028
test_tail_share: 0.8972


counts: {'overall': 7102, 'head': 730, 'tail': 6372}
[overall] {'HR@10': 0.042241622078287806, 'NDCG@10': 0.01999429146563426, 'HR@50': 0.23936919177696422, 'NDCG@50': 0.06550035783845347}
[head] {'HR@10': 0.0, 'NDCG@10': 0.0, 'HR@50': 1.095890410958904, 'NDCG@50': 0.2510415937369216}
[tail] {'HR@10': 0.047080979284369114, 'NDCG@10': 0.022284911799895564, 'HR@50': 0.14124293785310735, 'NDCG@50': 0.04424406433470555}
[long_tail] {'CatalogCoverage@10': 0.04525569467491326, 'TailCoverage@10': 0.04266471627963674, 'AvgPopularity@10': 3.109376966817976, 'TailRatio@10': 99.55927907631653, 'Personalization@10': 14.56535030564945, 'CatalogCoverage@50': 0.25946598280283606, 'TailCoverage@50': 0.24989333820930093, 'AvgPopularity@50': 2.857488557061844, 'TailRatio@50': 94.32638693325823, 'Personalization@50': 7.885432662436509}

ClassBalance | seed=4 | head_fraction=0.05 (5.000%)
num_head_items: 1,657
num_tail_items: 31,488
test_head_share: 0.2837
test_tail_share: 0.7163


counts: {'overall': 7102, 'head': 2015, 'tail': 5087}
[overall] {'HR@10': 0.042241622078287806, 'NDCG@10': 0.01999429146563426, 'HR@50': 0.23936919177696422, 'NDCG@50': 0.06550035783845347}
[head] {'HR@10': 0.09925558312655086, 'NDCG@10': 0.039159544730416274, 'HR@50': 0.6451612903225806, 'NDCG@50': 0.1648924651661602}
[tail] {'HR@10': 0.019657951641438964, 'NDCG@10': 0.012402786584852712, 'HR@50': 0.07863180656575586, 'NDCG@50': 0.026130376264769745}
[long_tail] {'CatalogCoverage@10': 0.04525569467491326, 'TailCoverage@10': 0.038109756097560975, 'AvgPopularity@10': 3.109376966817976, 'TailRatio@10': 83.57504928189242, 'Personalization@10': 14.56535030564945, 'CatalogCoverage@50': 0.25946598280283606, 'TailCoverage@50': 0.24136178861788618, 'AvgPopularity@50': 2.857488557061844, 'TailRatio@50': 86.383272317657, 'Personalization@50': 7.885432662436509}

ClassBalance | seed=4 | head_fraction=0.2 (20.000%)
num_head_items: 6,629
num_tail_items: 26,516
test_head_share: 0.5338
test_tail_shar

counts: {'overall': 7102, 'head': 3791, 'tail': 3311}
[overall] {'HR@10': 0.042241622078287806, 'NDCG@10': 0.01999429146563426, 'HR@50': 0.23936919177696422, 'NDCG@50': 0.06550035783845347}
[head] {'HR@10': 0.052756528620416784, 'NDCG@10': 0.020814160546501922, 'HR@50': 0.3692957003429174, 'NDCG@50': 0.09332422717316868}
[tail] {'HR@10': 0.030202355783751134, 'NDCG@10': 0.019055564891919587, 'HR@50': 0.0906070673512534, 'NDCG@50': 0.03364282577928542}
[long_tail] {'CatalogCoverage@10': 0.04525569467491326, 'TailCoverage@10': 0.03017046311660884, 'AvgPopularity@10': 3.109376966817976, 'TailRatio@10': 49.6972683751056, 'Personalization@10': 14.56535030564945, 'CatalogCoverage@50': 0.25946598280283606, 'TailCoverage@50': 0.23382108915371852, 'AvgPopularity@50': 2.857488557061844, 'TailRatio@50': 72.19008729935229, 'Personalization@50': 7.885432662436509}


,method,seed,lr,dropout,weight_decay,cb_beta,best_val_HR@50,test_overall_HR@10,test_overall_NDCG@10,test_overall_HR@50,...,test_TailRatio@10,test_Personalization@10,test_CatalogCoverage@50,test_TailCoverage@50,test_AvgPopularity@50,test_TailRatio@50,test_Personalization@50,test_count_overall,test_count_head,test_count_tail
0,ClassBalance,0,0.001,0.9,0.001,0.9999,0.352014,0.112644,0.076568,0.380175,...,30.678682,2.524920,0.217227,0.158395,3.300020,50.680653,1.497007,7102,3791,3311
1,ClassBalance,1,0.001,0.9,0.001,0.9999,0.154886,0.028161,0.028161,0.098564,...,53.155449,40.431116,0.310756,0.279077,2.847713,67.880034,29.851303,7102,3791,3311
2,ClassBalance,2,0.001,0.9,0.001,0.9999,0.253450,0.042242,0.022006,0.239369,...,35.161926,7.868929,0.307739,0.248906,2.847726,62.289496,5.786461,7102,3791,3311
3,ClassBalance,3,0.001,0.9,0.001,0.9999,0.140805,0.000000,0.000000,0.042242,...,79.321318,37.447400,0.301705,0.324332,2.470128,85.327232,3.981884,7102,3791,3311
4,ClassBalance,4,0.001,0.9,0.001,0.9999,0.366094,0.042242,0.019994,0.239369,...,49.697268,14.565350,0.259466,0.233821,2.857489,72.190087,7.885433,7102,3791,3311


## 11. Compact final table

In [14]:

def make_final_summary_table(
    metrics_df: pd.DataFrame,
    sweep_df: pd.DataFrame,
    method_name: str,
    save_path: str | None = None,
) -> pd.DataFrame:
    """
    One compact final table: one row = method × head_fraction.
    Metrics are aggregated over seeds as mean ± std.
    """
    if len(sweep_df) == 0:
        raise ValueError("sweep_df is empty")

    selected_metrics = [
        "overall_HR@50",
        "overall_NDCG@50",
        "head_HR@50",
        "head_NDCG@50",
        "tail_HR@50",
        "tail_NDCG@50",
        "CatalogCoverage@50",
        "TailCoverage@50",
        "AvgPopularity@50",
        "TailRatio@50",
        "Personalization@50",
    ]

    rows = []

    for head_fraction, group in sweep_df.groupby("head_fraction"):
        group = group.copy()

        row = {
            "method": method_name,
            "head_share": f"{100 * float(head_fraction):.3f}%",
            "head_fraction": float(head_fraction),
            "num_seeds": group["seed"].nunique() if "seed" in group.columns else len(group),
            "num_head_items": int(group["num_head_items"].iloc[0]) if "num_head_items" in group.columns else np.nan,
            "num_tail_items": int(group["num_tail_items"].iloc[0]) if "num_tail_items" in group.columns else np.nan,
        }

        if "test_head_share" in group.columns:
            row["test_head_share"] = f"{100 * float(group['test_head_share'].mean()):.2f}%"

        if "test_tail_share" in group.columns:
            row["test_tail_share"] = f"{100 * float(group['test_tail_share'].mean()):.2f}%"

        if metrics_df is not None and len(metrics_df) > 0:
            for hp_col in ["lr", "dropout", "weight_decay", "logq_weight", "cb_beta"]:
                if hp_col in metrics_df.columns:
                    vals = metrics_df[hp_col].dropna().unique()
                    if len(vals) == 1:
                        row[hp_col] = vals[0]
                    elif len(vals) > 1:
                        row[hp_col] = ", ".join(map(str, vals))

            if "best_val_HR@50" in metrics_df.columns:
                vals = metrics_df["best_val_HR@50"].dropna().to_numpy(dtype=float)
                if len(vals) > 0:
                    mean = float(np.mean(vals))
                    std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
                    row["best_val_HR@50"] = f"{mean:.2f} ± {std:.2f}"

        for metric in selected_metrics:
            if metric not in group.columns:
                continue

            vals = group[metric].dropna().to_numpy(dtype=float)

            if len(vals) == 0:
                row[metric] = "nan"
                continue

            mean = float(np.mean(vals))
            std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0

            if "AvgPopularity" in metric:
                row[metric] = f"{mean:.4f} ± {std:.4f}"
            else:
                row[metric] = f"{mean:.2f} ± {std:.2f}"

        rows.append(row)

    summary_df = pd.DataFrame(rows).sort_values("head_fraction").reset_index(drop=True)

    first_cols = [
        "method",
        "head_share",
        "head_fraction",
        "num_seeds",
        "num_head_items",
        "num_tail_items",
        "test_head_share",
        "test_tail_share",
        "lr",
        "dropout",
        "weight_decay",
        "logq_weight",
        "cb_beta",
        "best_val_HR@50",
    ]

    metric_cols = selected_metrics
    ordered_cols = [c for c in first_cols + metric_cols if c in summary_df.columns]
    other_cols = [c for c in summary_df.columns if c not in ordered_cols]
    summary_df = summary_df[ordered_cols + other_cols]

    if save_path is not None:
        summary_df.to_csv(save_path, index=False)
        print(f"saved: {save_path}")

    return summary_df

In [15]:
summary_df = make_final_summary_table(metrics_df, sweep_df, method_name="ClassBalance", save_path="yambda_classbalance_summary.csv")
summary_df

saved: yambda_classbalance_summary.csv


,method,head_share,head_fraction,num_seeds,num_head_items,num_tail_items,test_head_share,test_tail_share,lr,dropout,...,overall_NDCG@50,head_HR@50,head_NDCG@50,tail_HR@50,tail_NDCG@50,CatalogCoverage@50,TailCoverage@50,AvgPopularity@50,TailRatio@50,Personalization@50
0,ClassBalance,0.100%,0.001,5,33,33112,2.73%,97.27%,0.001,0.9,...,0.06 ± 0.04,0.72 ± 1.34,0.16 ± 0.30,0.19 ± 0.13,0.06 ± 0.04,0.28 ± 0.04,0.28 ± 0.04,2.8646 ± 0.2940,99.26 ± 1.01,9.80 ± 11.45
1,ClassBalance,0.500%,0.005,5,165,32980,6.39%,93.61%,0.001,0.9,...,0.06 ± 0.04,0.79 ± 0.79,0.39 ± 0.43,0.16 ± 0.12,0.04 ± 0.03,0.28 ± 0.04,0.28 ± 0.04,2.8646 ± 0.2940,96.92 ± 2.89,9.80 ± 11.45
2,ClassBalance,1.000%,0.010,5,331,32814,10.28%,89.72%,0.001,0.9,...,0.06 ± 0.04,0.55 ± 0.55,0.26 ± 0.29,0.16 ± 0.12,0.04 ± 0.03,0.28 ± 0.04,0.27 ± 0.05,2.8646 ± 0.2940,95.51 ± 3.70,9.80 ± 11.45
3,ClassBalance,5.000%,0.050,5,1657,31488,28.37%,71.63%,0.001,0.9,...,0.06 ± 0.04,0.40 ± 0.38,0.14 ± 0.14,0.12 ± 0.11,0.03 ± 0.03,0.28 ± 0.04,0.26 ± 0.05,2.8646 ± 0.2940,86.53 ± 6.82,9.80 ± 11.45
4,ClassBalance,20.000%,0.200,5,6629,26516,53.38%,46.62%,0.001,0.9,...,0.06 ± 0.04,0.28 ± 0.22,0.09 ± 0.08,0.11 ± 0.08,0.03 ± 0.02,0.28 ± 0.04,0.25 ± 0.06,2.8646 ± 0.2940,67.67 ± 12.75,9.80 ± 11.45
